In [ ]:
# MATLAB section 1/4 for PPThinning: Simulate PP via thinning

# % Simulate PP via thinning
# Given a conditional intensity function, we generate a point process
# consistent with this CIF.
#

# Python translation bootstrap for this helpfile.

# Topic: PPThinning
# Execution group: full
# Workflow family: network
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/PPThinning.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "PPThinning"
FAMILY = "network"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"PPThinning: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"PPThinning: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"PPThinning: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"PPThinning: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 5

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)


In [ ]:
# MATLAB section 2/4 for PPThinning: Basic Example

# % Basic Example
# MATLAB: close all;
# MATLAB: delta = 0.001;
# MATLAB: Tmax = 100;
# MATLAB: time = 0:delta:Tmax;
# MATLAB: f=.1;
# MATLAB: lambdaData = 10*sin(2*pi*f*time)+10; %lambda >=0
# MATLAB: lambda = Covariate(time,lambdaData, '\Lambda(t)','time','s','Hz',{'\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});
#
# MATLAB: lambdaBound = max(lambda);
# MATLAB: N=lambdaBound*(1.5*Tmax);   %Expected number of arrivals in interval 1.5*Tmax
# MATLAB: u = rand(1,N);              %N samples uniform(0,1)
# MATLAB: w = -log(u)./(lambdaBound); %N samples exponential rate lambdaBound (ISIs)
#
# MATLAB: tSpikes = cumsum(w);        %Spiketimes;
# MATLAB: tSpikes = tSpikes(tSpikes<=Tmax);%Spiketimes within Tmax
#
# Thinning
#
# MATLAB: lambdaRatio = lambda.getValueAt(tSpikes)./lambdaBound;
# lambdaRatio <=1
#
# draw uniform random number in 0,1
# MATLAB: u2 = rand(length(lambdaRatio),1);
#
# keep spike if lambda ratio is greater than random number
# MATLAB: tSpikesThin  = tSpikes(lambdaRatio>=u2);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/4 for PPThinning: Compare Constant rate process vs. thinned process

# % Compare Constant rate process vs. thinned process
# MATLAB: figure(1);
# MATLAB: n1 = nspikeTrain(tSpikes);
# MATLAB: n2 = nspikeTrain(tSpikesThin);
# MATLAB: subplot(2,2,1); n1.plot; plot(tSpikes,ones(size(tSpikes)),'.');
# MATLAB: v=axis; axis([0 Tmax/4 v(3) v(4)]);
# MATLAB: subplot(2,2,2); n1.plotISIHistogram;
# MATLAB: subplot(2,2,3); n2.plot; plot(tSpikes,ones(size(tSpikes)),'.');
# MATLAB: v=axis; axis([0 Tmax/4 v(3) v(4)]);
# MATLAB: subplot(2,2,4); n2.plotISIHistogram;
#
# MATLAB: figure(2);
# MATLAB: n2.plot;
# MATLAB: scaledProb = lambda*(1./lambdaBound);
# MATLAB: scaledProb.plot;
# MATLAB: v=axis;
# MATLAB: axis([0 Tmax/4 v(3) v(4)]);
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure(1);
fig = FIGURE_TRACKER.new_figure(section_index=3, matlab_line_number=34, matlab_snippet="figure(1);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPThinning.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 3, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: figure(2);
fig = FIGURE_TRACKER.new_figure(section_index=3, matlab_line_number=44, matlab_snippet="figure(2);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPThinning_01.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 3, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/4 for PPThinning: Simulate multiple realizations of a point process via thinning

# % Simulate multiple realizations of a point process via thinning
# The CIF class can generated realizations of a point process given
# a conditional intensity function (defined as a Covariate or SignalObj)
#
# MATLAB: numRealizations = 20;
# MATLAB: spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);
# MATLAB: figure(3);
# MATLAB: spikeColl.plot;
# MATLAB: lambda.plot;
# MATLAB: v=axis;
# MATLAB: axis([0 Tmax/4 v(3) v(4)]);
#
# Parity contract scalars for MATLAB/Python verification.
# MATLAB: parity = struct();
# MATLAB: parity.num_realizations = numRealizations;
#
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure(3);
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=57, matlab_snippet="figure(3);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPThinning_02.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 4, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #4 for PPThinning>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=51, matlab_snippet="<synthetic MATLAB figure event #4 for PPThinning>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPThinning_03.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=4 + 4, title=f"{TOPIC} Figure 004")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #5 for PPThinning>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=51, matlab_snippet="<synthetic MATLAB figure event #5 for PPThinning>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PPThinning_04.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=5 + 4, title=f"{TOPIC} Figure 005")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "delta = 0.001;",
    "Tmax = 100;",
    "time = 0:delta:Tmax;",
    "f=.1;",
    "lambdaData = 10*sin(2*pi*f*time)+10; %lambda >=0",
    "lambda = Covariate(time,lambdaData, '\\Lambda(t)','time','s','Hz',{'\\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});",
    "lambdaBound = max(lambda);",
    "N=lambdaBound*(1.5*Tmax);   %Expected number of arrivals in interval 1.5*Tmax",
    "u = rand(1,N);              %N samples uniform(0,1)",
    "w = -log(u)./(lambdaBound); %N samples exponential rate lambdaBound (ISIs)",
    "tSpikes = cumsum(w);        %Spiketimes;",
    "tSpikes = tSpikes(tSpikes<=Tmax);%Spiketimes within Tmax",
    "lambdaRatio = lambda.getValueAt(tSpikes)./lambdaBound;",
    "u2 = rand(length(lambdaRatio),1);",
    "tSpikesThin  = tSpikes(lambdaRatio>=u2);",
    "figure(1);",
    "n1 = nspikeTrain(tSpikes);",
    "n2 = nspikeTrain(tSpikesThin);",
    "subplot(2,2,1); n1.plot; plot(tSpikes,ones(size(tSpikes)),'.');",
    "v=axis; axis([0 Tmax/4 v(3) v(4)]);",
    "subplot(2,2,2); n1.plotISIHistogram;",
    "subplot(2,2,3); n2.plot; plot(tSpikes,ones(size(tSpikes)),'.');",
    "v=axis; axis([0 Tmax/4 v(3) v(4)]);",
    "subplot(2,2,4); n2.plotISIHistogram;",
    "figure(2);",
    "n2.plot;",
    "scaledProb = lambda*(1./lambdaBound);",
    "scaledProb.plot;",
    "v=axis;",
    "axis([0 Tmax/4 v(3) v(4)]);",
    "numRealizations = 20;",
    "spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);",
    "figure(3);",
    "spikeColl.plot;",
    "lambda.plot;",
    "v=axis;",
    "axis([0 Tmax/4 v(3) v(4)]);",
    "parity = struct();",
    "parity.num_realizations = numRealizations;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for PPThinning.")

# PPThinning: fixture-backed thinning acceptance parity.
from pathlib import Path
import nstat
from scipy.io import loadmat

m = loadmat(Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/PPThinning_gold.mat", squeeze_me=True)
time = np.asarray(m["time_pt"], dtype=float).reshape(-1); lambda_data = np.asarray(m["lambda_pt"], dtype=float).reshape(-1)
t_spikes = np.asarray(m["candidate_spikes_pt"], dtype=float).reshape(-1); lambda_ratio = np.asarray(m["lambda_ratio_pt"], dtype=float).reshape(-1); u2 = np.asarray(m["uniform_u2_pt"], dtype=float).reshape(-1)
expected = np.asarray(m["accepted_spikes_pt"], dtype=float).reshape(-1)
accepted = t_spikes[lambda_ratio >= u2]

fig, ax = plt.subplots(2, 1, figsize=(9, 5.6), sharex=False)
ax[0].vlines(t_spikes, 0.0, 1.0, color="0.5", linewidth=0.4, label="candidate")
ax[0].vlines(accepted, 0.0, 1.0, color="k", linewidth=0.6, label="accepted")
ax[0].set_xlim(0.0, float(np.asarray(m["tmax_pt"]).reshape(-1)[0]) / 4.0); ax[0].set_title("Candidate vs accepted spikes"); ax[0].legend(loc="upper right")
ax[1].plot(time, lambda_data, "b", linewidth=1.0); ax[1].set_xlim(0.0, float(np.asarray(m["tmax_pt"]).reshape(-1)[0]) / 4.0); ax[1].set_title("Conditional intensity"); ax[1].set_xlabel("time [s]")
plt.tight_layout(); plt.show()

assert accepted.shape == expected.shape
assert np.allclose(accepted, expected, atol=0.0)
assert np.all(np.diff(accepted) >= 0.0)
accept_ratio = float(accepted.size / max(t_spikes.size, 1)); expected_ratio = float(np.asarray(m["accept_ratio_pt"], dtype=float).reshape(-1)[0])
assert np.isclose(accept_ratio, expected_ratio, atol=0.0)

CHECKPOINT_METRICS = {
    "accepted_spike_count": float(accepted.size),
    "accept_ratio": float(accept_ratio),
    "lambda_mean": float(np.mean(lambda_data)),
}
CHECKPOINT_LIMITS = {
    "accepted_spike_count": (1.0, 1.0e7),
    "accept_ratio": (0.0, 1.0),
    "lambda_mean": (0.0, 1.0e6),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
